# InfiniSplat on Colab: from images to 3D Gaussian scenes

<a href="https://colab.research.google.com/github/Cortex78/InfiniSplat/blob/main/notebooks/InfiniSplat_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook installs and runs the inference code in [this repository](https://github.com/Cortex78/InfiniSplat):

| Part | Model | Input | Output |
| --- | --- | --- | --- |
| **A** | [InfiniSplat v2](https://zju3dv.github.io/InfiniSplat2/) · multi-view | 2–50 overlapping photos, with or without camera poses | Gaussian `scene.ply` + standalone HTML viewer |
| **B** | [InfiniSplat v1](https://zju3dv.github.io/InfiniSplat) · single image | One RGB image, optionally with a depth-sensor map | Gaussian `.ply` + HTML viewer (+ optional MP4) |

Run **Setup** first, then Part A, Part B, or both. Each part has its own Python environment, as the repository's install guides require, so the two never conflict.

**Before you start:** choose *Runtime → Change runtime type → A100 GPU*. The notebook is set up for an A100.

**Everything runs on the GPU:**
* The install cells check that PyTorch can use CUDA, and print the measured bf16 throughput and whether FlashAttention is available.
* Inference is always pinned to `cuda:0`. If there is no usable GPU, a run fails instead of quietly falling back to the CPU.
* Every run reports the peak GPU memory and utilisation it reached.

| GPU | Default v2 size | Notes |
| --- | --- | --- |
| **A100** (40 or 80 GB), H100 | every view, up to the model's limit of 50, at 518 px | |
| L4 24 GB | 24 views at 518 px | |
| T4 16 GB | 6 views at 392 px | No native bfloat16 or FlashAttention, so it is slower and runs out of memory sooner. |

With the automatic sizes, a v2 run that runs out of GPU memory is retried with fewer views.

First-time setup downloads roughly 5 GB of PyTorch/CUDA packages per part, plus the weights (6.6 GB for v2, 2.7–3.1 GB per v1 checkpoint). Loading the v2 model needs about **10.5 GB of system RAM**; A100 runtimes have plenty.

The reconstructions are feed-forward: there is no per-scene optimisation, so each scene takes seconds on an A100 once the model has loaded.

> **License:** InfiniSplat uses the [Project Registration License (PRL) v1.0](https://github.com/Cortex78/InfiniSplat/blob/main/LICENSE). Academic research, education, evaluation and personal use are free without registration; project use by companies, universities or other organisations requires a free prior registration. Third-party components keep their own licenses.

## 0 · Setup

In [ ]:
#@title 0.1 · Check the runtime (GPU, RAM, disk)
import os, re, shutil, subprocess, sys
from pathlib import Path

query = None
if shutil.which('nvidia-smi'):
    query = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version',
                            '--format=csv,noheader,nounits'], capture_output=True, text=True)
if query is None or query.returncode != 0 or not query.stdout.strip():
    raise SystemExit('No NVIDIA GPU found. Choose Runtime > Change runtime type > A100 GPU, then run this cell again.')
gpu_name, vram_mib, compute_cap, driver = [s.strip() for s in query.stdout.strip().splitlines()[0].split(',')]
GPU_NAME, VRAM_GB, COMPUTE_CAP = gpu_name, float(vram_mib) / 1024, float(compute_cap)
# The newest CUDA runtime this driver supports, from the nvidia-smi banner; it picks the PyTorch build.
cuda_banner = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
DRIVER_CUDA = (int(cuda_banner[1]), int(cuda_banner[2])) if cuda_banner else (0, 0)
RAM_GB = int(Path('/proc/meminfo').read_text().split()[1]) / 1024**2
BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
DISK_FREE_GB = shutil.disk_usage(BASE).free / 1024**3

print(f'GPU : {GPU_NAME}, {VRAM_GB:.0f} GB, compute capability {COMPUTE_CAP}, '
      f'driver {driver} (CUDA {DRIVER_CUDA[0]}.{DRIVER_CUDA[1]})')
print(f'RAM : {RAM_GB:.1f} GB   disk free: {DISK_FREE_GB:.0f} GB   Python {sys.version.split()[0]}')
if 'A100' in GPU_NAME:
    print('A100 detected: the notebook uses full-resolution defaults.')
else:
    print('Note: this notebook is tuned for an A100 (Runtime > Change runtime type > A100 GPU); '
          'it picks smaller defaults for this GPU.')
if COMPUTE_CAP < 8.0:
    print('Note: no native bfloat16/FlashAttention on this GPU, so inference is slower.')
if RAM_GB < 12:
    print('Warning: loading the v2 model needs about 10.5 GB of system RAM; use a High-RAM runtime for Part A.')
if DISK_FREE_GB < 40:
    print('Warning: Parts A and B together need about 35 GB of free disk.')

In [ ]:
#@title 0.2 · Get the code
REPO_URL = 'https://github.com/Cortex78/InfiniSplat'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

REPO = BASE / 'InfiniSplat'
if (REPO / '.git').is_dir():
    print(f'Reusing the existing clone at {REPO} (delete it to clone a different branch).')
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
print(subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--format=%h %s (%cd)'],
                     capture_output=True, text=True).stdout)

In [ ]:
#@title 0.3 · Helper functions (run once; expand to read the code) { display-mode: "form" }
import codecs, json, shlex, socket, threading, time, urllib.parse, zipfile
import numpy as np
from PIL import Image, ImageOps
from IPython.display import HTML, display

try:
    import plyfile  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plyfile'], check=True)

IN_COLAB = 'google.colab' in sys.modules
ENVS, OUTPUTS = BASE / 'envs', BASE / 'outputs'
V2_DIR, V1_DIR = REPO / 'InfiniSplat_v2', REPO / 'InfiniSplat_v1'
V2_WEIGHTS = V2_DIR / 'weights' / 'da3'
V2_PY, V1_PY = ENVS / 'v2' / 'bin' / 'python', ENVS / 'v1' / 'bin' / 'python'
OUTPUTS.mkdir(parents=True, exist_ok=True)
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
RUN_INPUTS = {}  # output directory -> input image paths, used by the previews

# Starting sizes for v2 by GPU class (see the table at the top). Activations grow by roughly 0.2 GB
# per view at 518 px (measured in fp32 on CPU) on top of 6.6 GB of weights, so 50 views should need
# about 20 GB. With these automatic sizes, a run that runs out of GPU memory is retried with fewer views.
if COMPUTE_CAP < 8.0 or VRAM_GB < 20:
    V2_AUTO = dict(views=6, long_side=392)  # T4: no FlashAttention, so attention memory grows quadratically
elif VRAM_GB < 35:
    V2_AUTO = dict(views=24, long_side=518)  # L4
else:
    V2_AUTO = dict(views=50, long_side=518)  # A100, H100: every view the model accepts


def env_with(*bin_dirs, **extra):
    """Environment overrides that put the given directories first on PATH."""
    return {'PATH': os.pathsep.join([*map(str, bin_dirs), os.environ['PATH']]), **extra}


# Each version runs from its own virtualenv; node comes from the nodejs-wheel package in that env.
V2_ENV = env_with(ENVS / 'splat-v2' / 'node_modules' / '.bin', ENVS / 'v2' / 'bin',
                  PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
V1_ENV = env_with(ENVS / 'v1' / 'bin',
                  SPLAT_TRANSFORM=str(ENVS / 'splat-v1' / 'node_modules' / '.bin' / 'splat-transform'))


def torch_index(choice):
    """PyTorch wheel tag and index: CUDA 12.8 (the tested builds) if the driver supports it, else 12.6."""
    tag = choice if choice != 'auto' else ('cu128' if DRIVER_CUDA >= (12, 8) else 'cu126')
    return tag, f'https://download.pytorch.org/whl/{tag}'


# Run inside an environment: fail unless PyTorch can compute on the GPU, and show what it gets.
CUDA_CHECK = r"""
import time, torch, torch.nn.functional as F
if not torch.cuda.is_available():
    raise SystemExit(f"PyTorch {torch.__version__} (CUDA {torch.version.cuda}) cannot use the GPU. Check the "
                     "runtime type; with an old driver, rerun the install cell with TORCH_CUDA = cu126.")
props = torch.cuda.get_device_properties(0)
x = torch.randn(8192, 8192, device="cuda", dtype=torch.bfloat16)
x @ x
torch.cuda.synchronize()
start = time.time()
for _ in range(5):
    x @ x
torch.cuda.synchronize()
tflops = 5 * 2 * 8192**3 / (time.time() - start) / 1e12
try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
    q = torch.randn(1, 8, 2048, 64, device="cuda", dtype=torch.bfloat16)
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        F.scaled_dot_product_attention(q, q, q)
    flash = "yes"
except RuntimeError:
    flash = "no"
print(f"CUDA OK: torch {torch.__version__} (CUDA {torch.version.cuda}) on {props.name}, "
      f"{props.total_memory / 1024**3:.0f} GB, compute {props.major}.{props.minor}; "
      f"bf16 matmul {tflops:.0f} TFLOPS; FlashAttention: {flash}")
"""


def check_cuda(python):
    sh(f'{shlex.quote(str(python))} -c {shlex.quote(CUDA_CHECK)}')


class GpuMonitor:
    """Sample nvidia-smi while a run is in progress and report how much of the GPU it used."""

    def __init__(self, interval=0.25):
        self.interval, self.idle_gb, self.peak_gb, self.peak_util = interval, 0.0, 0.0, 0.0

    @staticmethod
    def sample():
        out = subprocess.run(['nvidia-smi', '-i', '0', '--query-gpu=memory.used,utilization.gpu',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout
        used, util = out.strip().splitlines()[0].split(',')
        return float(used) / 1024, float(util)

    def __enter__(self):
        try:
            self.idle_gb = self.sample()[0]
        except (ValueError, IndexError, OSError):
            pass
        self.done = threading.Event()

        def loop():
            while True:
                try:
                    used, util = self.sample()
                    self.peak_gb, self.peak_util = max(self.peak_gb, used), max(self.peak_util, util)
                except (ValueError, IndexError, OSError):
                    pass
                if self.done.wait(self.interval):
                    break

        self.thread = threading.Thread(target=loop, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, *exc):
        self.done.set()
        self.thread.join()
        print(f'GPU usage: {GPU_NAME}, peak memory {self.peak_gb:.1f} of {VRAM_GB:.0f} GB, '
              f'peak utilisation {self.peak_util:.0f}%')
        if exc[0] is None and self.peak_gb - self.idle_gb < 1:
            print('WARNING: this run barely used the GPU. Rerun the install cell and check its "CUDA OK" line.')


class CommandError(RuntimeError):
    def __init__(self, code, tail):
        low = tail.lower()
        self.code, self.oom = code, 'out of memory' in low
        if code in (-9, 137):
            hint = ('the process was killed, most likely for lack of system RAM. Loading the v2 model needs '
                    'about 10.5 GB; use a High-RAM runtime or restart the runtime to free memory.')
        elif 'out of memory' in low:
            hint = 'the GPU ran out of memory. Use fewer views and/or a smaller long side, then run again.'
        elif 'no space left' in low:
            hint = f'the disk is full. Delete old results in {OUTPUTS} or start a fresh runtime.'
        else:
            hint = 'see the output above.'
        super().__init__(f'Command failed with exit code {code}: {hint}')


def sh(cmd, cwd=None, env=None, show=None):
    """Run a bash command, stream its output, and raise CommandError if it fails."""
    print(f'$ {show or cmd}', flush=True)
    proc = subprocess.Popen(['bash', '-c', 'set -eo pipefail\n' + cmd], cwd=cwd,
                            env={**os.environ, **(env or {})},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    decoder, tail = codecs.getincrementaldecoder('utf-8')('replace'), ''
    for chunk in iter(lambda: proc.stdout.read1(65536), b''):
        text = decoder.decode(chunk)
        sys.stdout.write(text)
        sys.stdout.flush()
        tail = (tail + text)[-20000:]
    code = proc.wait()
    if code:
        raise CommandError(code, tail)


def uv_venv(path, python):
    """Create a fresh virtualenv with uv, installing uv into the kernel first if needed."""
    if not shutil.which('uv'):
        sh(f'{shlex.quote(sys.executable)} -m pip install -q uv')
    shutil.rmtree(path, ignore_errors=True)
    sh(f'uv venv --python {python} {shlex.quote(str(path))}')


def require(path, cell):
    if not Path(path).exists():
        raise FileNotFoundError(f'{path} is missing; run cell {cell} first.')


def pick_views(paths, count):
    """Indices of `count` evenly spaced items, keeping the first and last."""
    if count >= len(paths):
        return list(range(len(paths)))
    return sorted(set(np.linspace(0, len(paths) - 1, count).round().astype(int).tolist()))


def list_images(directory, exts=IMAGE_EXTS):
    return sorted((p for p in Path(directory).iterdir() if p.is_file() and p.suffix.lower() in exts),
                  key=lambda p: p.name)


def upload_files(kind):
    """Upload files through the browser into a new directory under OUTPUTS/uploads."""
    if not IN_COLAB:
        raise RuntimeError('Uploading needs Colab; set the path parameter instead.')
    from google.colab import files
    dest = OUTPUTS / 'uploads' / f'{kind}-{time.strftime("%Y%m%d-%H%M%S")}'
    dest.mkdir(parents=True)
    for name, data in files.upload().items():
        (dest / Path(name).name).write_bytes(data)
        Path(name).unlink(missing_ok=True)  # files.upload() also saves a copy in the working directory
    return dest


# ---------------------------------------------------------------- running the models
def run_v2(images, name, cameras=None, pose_mode='auto', long_side=518, html=True):
    """Run the `infinisplat` CLI on explicitly ordered images; return the new output directory."""
    images = [Path(p) for p in images]
    if not 2 <= len(images) <= 50:
        raise ValueError(f'InfiniSplat v2 takes 2-50 images; got {len(images)}.')
    mode = pose_mode if pose_mode != 'auto' else ('provided' if cameras is not None else 'da3-pose')
    if mode == 'provided' and cameras is None:
        raise ValueError('pose mode "provided" needs a camera file; choose da3-pose or pose-free instead.')
    require(V2_WEIGHTS / 'model.safetensors', 'A.2')
    out = OUTPUTS / 'v2' / f'{name}-{mode}-{len(images)}v-{long_side}px-{time.strftime("%Y%m%d-%H%M%S")}'
    argv = [ENVS / 'v2' / 'bin' / 'infinisplat', '--model', V2_WEIGHTS, '--output', out, '--device', 'cuda:0',
            '--pose-mode', mode, '--long-side', long_side, '--max-views', len(images)]
    if mode == 'provided':
        argv += ['--cameras', cameras]
    if not html:
        argv.append('--no-html')
    argv = [str(a) for a in argv + ['--images', *images]]
    start = time.time()
    with GpuMonitor():
        sh(shlex.join(argv), env=V2_ENV, show=shlex.join(argv[:-len(images)]) + f' <{len(images)} images>')
    print(f'Finished in {time.time() - start:.0f} s: {out}')
    RUN_INPUTS[str(out)] = [str(p) for p in images]
    return out


def reconstruct_v2(images, name, cameras_npz=None, pose_mode='auto', num_views=0, long_side=0, html=True):
    """Pick evenly spaced views (and their cameras) and run v2.

    With automatic sizes (num_views=0), a run that runs out of GPU memory is retried with fewer views.
    """
    auto = not num_views
    views, long_side = num_views or V2_AUTO['views'], long_side or V2_AUTO['long_side']
    use_cameras = cameras_npz is not None and pose_mode in ('auto', 'provided')
    while True:
        selected = pick_views(images, min(views, 50))
        names = [images[i].name for i in selected]
        shown = ', '.join(names) if len(names) <= 8 else ', '.join(names[:3]) + ', …, ' + names[-1]
        print(f'Using {len(selected)} of {len(images)} views at {long_side} px: {shown}')
        cameras = None
        if use_cameras:
            cameras = subset_cameras(cameras_npz, selected,
                                     OUTPUTS / 'v2_inputs' / f'{name}-{len(selected)}-views.npz')
        try:
            return run_v2([images[i] for i in selected], name, cameras=cameras, pose_mode=pose_mode,
                          long_side=long_side, html=html)
        except CommandError as error:
            if not (auto and error.oom and len(selected) > 2):
                raise
            views = max(2, len(selected) * 2 // 3)
            print(f'\nThe GPU ran out of memory with {len(selected)} views; retrying with {views}.\n')


def subset_cameras(npz, indices, dest):
    """Write the cameras of the selected views, in the same order as the selected images."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with np.load(npz, allow_pickle=False) as data:
        np.savez(dest, c2w=data['c2w'][indices], intrinsics=data['intrinsics'][indices])
    return dest


def run_v1(inputs, mode='rgb', focal_mm=0.0, prompt_depth=None):
    """Run v1 batch inference on one image or a directory; return the output root."""
    require(V1_DIR / 'checkpoints' / f'infinisplat_{mode}.ckpt', 'B.2')
    out = OUTPUTS / 'v1' / f'{mode}-{time.strftime("%Y%m%d-%H%M%S")}'
    # Pin the GPU: the script's default device ("auto") would silently fall back to the CPU.
    argv = [V1_PY, '-m', 'src.demo.infer_batch_images', '--mode', mode, '--device', 'cuda:0',
            '--input', Path(inputs).resolve(), '--output-dir', out]
    if focal_mm:
        argv += ['--focal-mm', focal_mm]
    if prompt_depth:
        argv += ['--prompt-depth', Path(prompt_depth).resolve()]
    start = time.time()
    with GpuMonitor():
        sh(shlex.join(map(str, argv)), cwd=V1_DIR, env=V1_ENV)
    print(f'Finished in {time.time() - start:.0f} s: {out}')
    return out


# ---------------------------------------------------------------- looking at Gaussians
SH_C0 = 0.28209479177387814


def load_gaussians(ply_path):
    """Means, display colours, opacities and single-property metadata elements of a Gaussian PLY."""
    from plyfile import PlyData
    ply = PlyData.read(str(ply_path))
    v = ply['vertex'].data
    xyz = np.stack([v['x'], v['y'], v['z']], 1).astype(np.float64)
    rgb = np.clip(0.5 + SH_C0 * np.stack([v[f'f_dc_{i}'] for i in range(3)], 1), 0, 1).astype(np.float32)
    alpha = 1 / (1 + np.exp(-v['opacity'].astype(np.float64)))
    extra = {e.name: np.asarray(e.data[e.properties[0].name]) for e in ply.elements
             if e.name != 'vertex' and len(e.properties) == 1}
    return xyz, rgb, alpha, extra


def render_points(xyz, rgb, alpha, c2w, K, size, min_alpha=0.1):
    """Nearest-point z-buffer projection of Gaussian centres (OpenCV camera). A quick preview, not a splat render."""
    W, H = size
    w2c = np.linalg.inv(c2w)
    keep = alpha > min_alpha
    p = xyz[keep] @ w2c[:3, :3].T + w2c[:3, 3]
    c = rgb[keep]
    front = p[:, 2] > 1e-6
    p, c = p[front], c[front]
    u = np.floor(K[0, 0] * p[:, 0] / p[:, 2] + K[0, 2] + 0.5).astype(np.int64)
    v = np.floor(K[1, 1] * p[:, 1] / p[:, 2] + K[1, 2] + 0.5).astype(np.int64)
    inside = (u >= 0) & (u < W) & (v >= 0) & (v < H)
    pixel, depth, c = v[inside] * W + u[inside], p[inside, 2], c[inside]
    order = np.lexsort((depth, pixel))  # group by pixel, nearest first
    pixel, c = pixel[order], c[order]
    first = np.r_[True, pixel[1:] != pixel[:-1]]
    image = np.full((H * W, 3), 0.4, np.float32)  # the HTML viewer's grey background
    image[pixel[first]] = c[first]
    return image.reshape(H, W, 3)


def orbit(c2w, pivot_depth, degrees):
    """Yaw a camera about the point `pivot_depth` in front of it."""
    t = np.radians(degrees)
    R = np.array([[np.cos(t), 0, np.sin(t)], [0, 1, 0], [-np.sin(t), 0, np.cos(t)]])
    pivot = np.array([0.0, 0.0, pivot_depth])
    local = np.eye(4)
    local[:3, :3], local[:3, 3] = R, pivot - R @ pivot
    return c2w @ local


def preview(gaussians, c2w, K, size, reference=None, title='', degrees=12, max_side=640):
    """Input image next to point projections from the input camera and two orbiting novel views."""
    import matplotlib.pyplot as plt
    xyz, rgb, alpha = gaussians
    scale = min(1.0, max_side / max(size))
    K = np.diag([scale, scale, 1.0]) @ K
    size = (max(1, round(size[0] * scale)), max(1, round(size[1] * scale)))
    cam = (xyz - c2w[:3, 3]) @ c2w[:3, :3]
    visible = (alpha > 0.1) & (cam[:, 2] > 1e-6)
    pivot = float(np.median(cam[visible, 2])) if visible.any() else 1.0
    panels = [] if reference is None else [('Input image', reference)]
    for label, deg in ((f'{-degrees}° orbit', -degrees), ('Input camera', 0), (f'+{degrees}° orbit', degrees)):
        panels.append((label, render_points(xyz, rgb, alpha, orbit(c2w, pivot, deg), K, size)))
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4 * size[1] / size[0] + 0.7))
    for ax, (label, image) in zip(np.atleast_1d(axes), panels):
        ax.imshow(image)
        ax.set_title(label)
        ax.axis('off')
    fig.suptitle(f'{title}\nGaussian centres projected as points (holes are expected; '
                 'the HTML viewer below renders the real splats)', fontsize=10)
    plt.tight_layout()
    plt.show()


def plot_3d(gaussians, title='', max_points=60_000):
    """Interactive point cloud of a random subset of Gaussian centres."""
    import plotly.graph_objects as go
    xyz, rgb, alpha = gaussians
    keep = np.flatnonzero(alpha > 0.1)
    if len(keep) > max_points:
        keep = np.random.default_rng(0).choice(keep, max_points, replace=False)
    p, c = xyz[keep], (rgb[keep] * 255).astype(np.uint8)
    lo, hi = np.percentile(p, [1, 99], axis=0)
    margin = (hi - lo) * 0.1
    inside = np.all((p >= lo - margin) & (p <= hi + margin), axis=1)  # drop far floaters from the view box
    p, c = p[inside], c[inside]
    hidden = dict(visible=False)
    fig = go.Figure(go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers', hoverinfo='skip',
                                 marker=dict(size=1.5, color=[f'rgb({r},{g},{b})' for r, g, b in c])))
    fig.update_layout(title=f'{title} ({len(p):,} of {len(xyz):,} Gaussians, drag to rotate)',
                      height=600, margin=dict(l=0, r=0, t=40, b=0), paper_bgcolor='#666666',
                      font=dict(color='white'),
                      scene=dict(aspectmode='data', xaxis=hidden, yaxis=hidden, zaxis=hidden, bgcolor='#666666',
                                 camera=dict(up=dict(x=0, y=-1, z=0), eye=dict(x=0, y=-0.4, z=-1.8))))
    fig.show()


# ---------------------------------------------------------------- viewers and downloads
VIEWER_PORT = 8765


def ensure_file_server():
    """Serve OUTPUTS over HTTP so Colab can proxy the HTML viewers and videos into the notebook."""
    def is_up():
        with socket.socket() as s:
            return s.connect_ex(('127.0.0.1', VIEWER_PORT)) == 0
    if is_up():
        return
    subprocess.Popen([sys.executable, '-m', 'http.server', str(VIEWER_PORT), '--directory', str(OUTPUTS)],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(100):
        time.sleep(0.1)
        if is_up():
            return
    raise RuntimeError('Could not start the local file server for the viewer.')


def served_path(path):
    return '/' + urllib.parse.quote(Path(path).resolve().relative_to(OUTPUTS.resolve()).as_posix())


def show_viewer(html_path, height=640):
    html_path = Path(html_path)
    if not html_path.is_file():
        print('No HTML viewer for this run (it was disabled, or splat-transform was unavailable).')
        return
    if not IN_COLAB:
        print('Open this file in a browser:', html_path)
        return
    from google.colab import output
    ensure_file_server()
    print(f'Interactive viewer ({html_path.stat().st_size / 1e6:.0f} MB; it can take a few seconds to load). '
          'Drag to orbit, scroll to zoom.')
    output.serve_kernel_port_as_iframe(VIEWER_PORT, path=served_path(html_path), height=height)
    output.serve_kernel_port_as_window(VIEWER_PORT, path=served_path(html_path),
                                       anchor_text='Open the viewer in a new browser tab')


def show_video(mp4_path):
    if not IN_COLAB:
        print('Video:', mp4_path)
        return
    from google.colab.output import eval_js
    ensure_file_server()
    url = eval_js(f'google.colab.kernel.proxyPort({VIEWER_PORT})').rstrip('/') + served_path(mp4_path)
    display(HTML(f'<video src="{url}" controls autoplay loop muted style="max-width:100%"></video>'))


def list_files(directory):
    for path in sorted(Path(directory).rglob('*')):
        if path.is_file():
            print(f'  {path.relative_to(directory)!s:40s} {path.stat().st_size / 1e6:8.1f} MB')


def download(path):
    """Download a file, or a directory as a zip, through the browser."""
    path = Path(path)
    if path.is_dir():
        path = Path(shutil.make_archive(str(path), 'zip', root_dir=path))
    if IN_COLAB:
        from google.colab import files
        files.download(str(path))
    else:
        print('Saved at', path)


def show_v2(out, html=True):
    """Summary, point previews, 3D plot and HTML viewer for one v2 output directory."""
    out = Path(out)
    meta = json.loads((out / 'result.json').read_text())
    with np.load(out / 'cameras.npz') as cams:  # effective cameras; intrinsics in processed-image pixels
        c2w, K = cams['c2w'][0].astype(np.float64), cams['intrinsics'][0].astype(np.float64)
    first = meta['inputs'][0]
    size = tuple(first['processed_wh'])
    stats = meta['stats']
    print(f"pose mode {meta['pose_mode']}, {len(meta['inputs'])} views at {size[0]}x{size[1]} px, "
          f"{stats['final_count']:,} Gaussians (from {stats['raw_count']:,} before fusion)")
    list_files(out)
    reference = None
    if str(out) in RUN_INPUTS:  # crop and resize the first input exactly as the pipeline did
        with Image.open(RUN_INPUTS[str(out)][0]) as im:
            reference = np.asarray(ImageOps.exif_transpose(im).convert('RGB').resize(
                size, Image.Resampling.BICUBIC, box=tuple(first['crop_box'])))
    xyz, rgb, alpha, _ = load_gaussians(out / 'scene.ply')
    preview((xyz, rgb, alpha), c2w, K, size, reference, title=out.name)
    plot_3d((xyz, rgb, alpha), title=out.name)
    if html:
        show_viewer(out / 'scene.html')


def load_depth(path):
    """Prompt depth as a dense array (0 where empty), for display only."""
    path = Path(path)
    if path.suffix == '.npy':
        return np.load(path)
    with np.load(path, allow_pickle=False) as data:
        if 'mask' in data and 'value' in data:
            depth = np.zeros(data['mask'].shape, np.float32)
            depth[data['mask']] = data['value']
            return depth
        return data[data.files[0]]


def show_v1(out, images, depth=None, html=True):
    """Point previews, 3D plot, HTML viewer and optional video for each case of a v1 run."""
    import matplotlib.pyplot as plt
    by_stem = {Path(p).stem: Path(p) for p in images}
    for case in sorted(p for p in Path(out).iterdir() if p.is_dir()):
        print(f'== {case.name}')
        list_files(case)
        if depth is not None:
            d = load_depth(depth)
            fig, ax = plt.subplots(figsize=(6, 6 * d.shape[0] / d.shape[1]))
            ax.imshow(np.ma.masked_equal(d, 0), cmap='turbo')
            ax.set_title(f'Prompt depth ({int((d > 0).sum()):,} valid pixels)')
            ax.axis('off')
            plt.show()
        xyz, rgb, alpha, extra = load_gaussians(case / f'{case.name}.ply')
        print(f'{len(xyz):,} Gaussians')
        with Image.open(by_stem[case.name]) as im:
            reference = np.asarray(ImageOps.exif_transpose(im).convert('RGB'))
        # The PLY stores fx for the 1536x1152 inference image; render at the input's aspect instead.
        height, width = reference.shape[:2]
        f = float(extra['intrinsic'][0]) * width / float(extra['image_size'][0])
        K = np.array([[f, 0, width / 2], [0, f, height / 2], [0, 0, 1]])
        w2c = np.asarray(extra['extrinsic'], np.float64).reshape(4, 4)
        preview((xyz, rgb, alpha), np.linalg.inv(w2c), K, (width, height), reference, title=case.name)
        plot_3d((xyz, rgb, alpha), title=case.name)
        if html:
            show_viewer(case / f'{case.name}.html')
        if (case / f'{case.name}.mp4').is_file():
            show_video(case / f'{case.name}.mp4')


print(f'Helpers ready. v2 starting size for this GPU: {V2_AUTO["views"]} views at {V2_AUTO["long_side"]} px.')

## Part A · InfiniSplat v2: multiple images → 3D Gaussian scene

v2 conditions [Depth Anything 3](https://github.com/ByteDance-Seed/Depth-Anything-3) features on cameras and decodes per-pixel Gaussians, then fuses them into one scene. It has three camera modes:

* `provided`: you supply camera poses and intrinsics (an NPZ or a COLMAP model);
* `da3-pose` (default without cameras): DA3 first predicts the cameras, then conditions reconstruction on them;
* `pose-free`: images only, skipping the camera prepass.

In [ ]:
#@title A.1 · Install the v2 environment with CUDA PyTorch (about 3–5 min)
TORCH_CUDA = 'auto'  #@param ["auto", "cu128", "cu126"]
#@markdown `auto` installs the CUDA 12.8 build that v2 was tested with when the driver supports it, otherwise the CUDA 12.6 build of the same PyTorch version. The cell ends with a `CUDA OK` line from a bf16 matmul on the GPU; it fails if PyTorch cannot use the GPU.

# Follows InfiniSplat_v2/INSTALL.md in an isolated env: v2 needs Python 3.10-3.12 and numpy<2.
tag, index = torch_index(TORCH_CUDA)
marker = ENVS / 'v2' / '.ready'
if not marker.is_file() or marker.read_text() != tag:
    uv_venv(ENVS / 'v2', '3.12')
    pip = f'uv pip install --python {V2_PY}'
    sh(f'{pip} torch==2.7.1 --index-url {index}')
    sh(f'{pip} {V2_DIR} nodejs-wheel==22.20.0')
    sh(f"{pip} --no-deps 'depth-anything-3 @ git+https://github.com/ByteDance-Seed/Depth-Anything-3.git"
       "@41736238f5bced4debf3f2a12375d2466874866d'")
    # The HTML exporter patches the viewer template of exactly this converter version.
    sh(f'npm install --prefix {ENVS / "splat-v2"} --no-audit --no-fund @playcanvas/splat-transform@2.6.0', env=V2_ENV)
    marker.write_text(tag)
check_cuda(V2_PY)
sh('infinisplat --help | sed -n 1,4p; splat-transform --version', env=V2_ENV)

In [ ]:
#@title A.2 · Download the v2 weights (6.6 GB) and the three example scenes (90 MB)
require(V2_PY, 'A.1')
if not (V2_WEIGHTS / 'model.safetensors').is_file():
    shutil.rmtree(V2_WEIGHTS, ignore_errors=True)
    sh(f'{V2_PY} scripts/download_weights.py --repo-id Edisoneh/InfiniSplat-v2 --output {V2_WEIGHTS}',
       cwd=V2_DIR, env=V2_ENV)
    # The script verifies the SHA256 and copies out of the Hugging Face cache; drop the cached copy to save disk.
    hf_home = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface'))
    shutil.rmtree(Path(os.environ.get('HF_HUB_CACHE', hf_home / 'hub')) / 'models--Edisoneh--InfiniSplat-v2',
                  ignore_errors=True)
if not (V2_DIR / 'examples' / 'data' / 'manifest.json').is_file():
    sh('hf download Edisoneh/InfiniSplat-v2 examples-png.zip --local-dir .', cwd=V2_DIR, env=V2_ENV)
    with zipfile.ZipFile(V2_DIR / 'examples-png.zip') as archive:  # keep the tracked examples/NOTICE.md as is
        archive.extractall(V2_DIR, members=[m for m in archive.namelist() if m.startswith('examples/data/')])
    (V2_DIR / 'examples-png.zip').unlink()
list_files(V2_WEIGHTS)
for scene in sorted(p for p in (V2_DIR / 'examples' / 'data').iterdir() if p.is_dir()):
    cams = 'with cameras' if (scene / 'cameras.npz').is_file() else 'no cameras'
    print(f'{scene.name:20s} {len(list_images(scene / "images"))} views, {cams}')

In [ ]:
#@title A.3 · Reconstruct an example scene
EXAMPLE = 'dtu_provided'  #@param ["dtu_provided", "hypersim_provided", "hypersim_da3_pose"]
POSE_MODE = 'auto'  #@param ["auto", "provided", "da3-pose", "pose-free"]
NUM_VIEWS = 0  #@param {type:"integer"}
LONG_SIDE = 0  #@param {type:"integer"}
EXPORT_HTML = True  #@param {type:"boolean"}
#@markdown * `auto` uses the example's cameras when it has them (`provided`), otherwise `da3-pose`. Try `pose-free` or `da3-pose` on a calibrated example to compare.
#@markdown * `NUM_VIEWS = 0` and `LONG_SIDE = 0` use this GPU's starting size (printed by cell 0.3; an A100 gets every view, up to 50, at 518 px). If that runs out of GPU memory, the run is retried with fewer views. Views are sampled evenly; `LONG_SIDE` accepts 112–2044 px (the model's default is 518).
#@markdown * With your own `NUM_VIEWS`, an out-of-memory run stops instead; lower `NUM_VIEWS` first, then `LONG_SIDE`.
#@markdown * Each run starts a fresh process that verifies the 6.6 GB checkpoint's SHA-256 and loads the model (about a minute) before reconstructing.

scene_dir = V2_DIR / 'examples' / 'data' / EXAMPLE
require(scene_dir, 'A.2')
cameras_npz = scene_dir / 'cameras.npz'
LAST_V2 = reconstruct_v2(list_images(scene_dir / 'images'), EXAMPLE,
                         cameras_npz=cameras_npz if cameras_npz.is_file() else None, pose_mode=POSE_MODE,
                         num_views=NUM_VIEWS, long_side=LONG_SIDE, html=EXPORT_HTML)

In [ ]:
#@title A.4 · Look at the result
show_v2(LAST_V2)

In [ ]:
#@title A.5 · Reconstruct your own photos
IMAGES_DIR = ''  #@param {type:"string"}
POSE_MODE = 'auto'  #@param ["auto", "da3-pose", "pose-free", "provided"]
CAMERAS_NPZ = ''  #@param {type:"string"}
NUM_VIEWS = 0  #@param {type:"integer"}
LONG_SIDE = 0  #@param {type:"integer"}
EXPORT_HTML = True  #@param {type:"boolean"}
#@markdown * Leave `IMAGES_DIR` empty to upload files, or give a folder (for a Google Drive folder, mount Drive first with the folder icon on the left). Use 2–50 overlapping JPG/PNG/WebP photos of one static scene; they are read in filename order. Convert HEIC photos to JPG first.
#@markdown * Without cameras, `auto` runs `da3-pose`. With `CAMERAS_NPZ` (`c2w` `[N,4,4]` OpenCV camera-to-world and `intrinsics` `[N,3,3]` in original-image pixels, in filename order) it runs `provided`. See `InfiniSplat_v2/docs/inference.md` and `docs/colmap.md`.
#@markdown * If there are more photos than `NUM_VIEWS` (or the GPU's starting size when `0`), an evenly spaced subset is used; with `0`, an out-of-memory run is retried with fewer views.

source = Path(IMAGES_DIR) if IMAGES_DIR else upload_files('v2')
images = list_images(source)
if len(images) < 2:
    raise ValueError(f'Found {len(images)} JPG/PNG/WebP images in {source}; v2 needs at least 2.')
LAST_V2 = reconstruct_v2(images, source.name, cameras_npz=CAMERAS_NPZ or None, pose_mode=POSE_MODE,
                         num_views=NUM_VIEWS, long_side=LONG_SIDE, html=EXPORT_HTML)
show_v2(LAST_V2, html=EXPORT_HTML)

In [ ]:
#@title A.6 · Download the latest v2 result
WHAT = 'everything (zip)'  #@param ["everything (zip)", "scene.ply", "scene.html"]
download(LAST_V2 if WHAT.startswith('everything') else LAST_V2 / WHAT)

## Part B · InfiniSplat v1: one image → 3D Gaussian scene

v1 predicts a Gaussian scene from a single RGB image (`rgb` checkpoint), optionally guided by a sparse or dense depth-sensor map such as LiDAR (`lidar` checkpoint). Images are processed at 1536×1152. Camera focal length comes from `--focal-mm`/`--focal-px`, the image's EXIF data, or a 30 mm full-frame-equivalent fallback.

In [ ]:
#@title B.1 · Install the v1 environment with CUDA PyTorch (about 4–6 min)
TORCH_CUDA = 'auto'  #@param ["auto", "cu128", "cu126"]
INSTALL_GSPLAT = False  #@param {type:"boolean"}
#@markdown * `auto` installs the CUDA 12.8 build from v1's install guide when the driver supports it, otherwise the CUDA 12.6 build of the same versions. The cell fails unless PyTorch and xFormers both run on the GPU.
#@markdown * `gsplat` is only needed for the optional MP4 fly-through video. It is compiled from source for this GPU, which takes roughly 10–20 minutes. PLY and HTML export work without it.

# Follows InfiniSplat_v1/INSTALL.md (Python 3.10, PyTorch 2.9.0, xFormers 0.0.33.post1) in an isolated env.
tag, index = torch_index(TORCH_CUDA)
marker = ENVS / 'v1' / '.ready'
if not marker.is_file() or marker.read_text() != tag:
    uv_venv(ENVS / 'v1', '3.10')
    pip = f'uv pip install --python {V1_PY}'
    sh(f'{pip} torch==2.9.0 torchvision==0.24.0 xformers==0.0.33.post1 --index-url {index}')
    sh(f'{pip} -r requirements.txt', cwd=V1_DIR)  # includes nodejs-wheel, which provides node and npm
    # The version used by v1's own web demo; the exporter patches its viewer template.
    sh(f'npm install --prefix {ENVS / "splat-v1"} --no-audit --no-fund @playcanvas/splat-transform@3.1.6', env=V1_ENV)
    marker.write_text(tag)
if INSTALL_GSPLAT and subprocess.run([str(V1_PY), '-c', 'import gsplat'], capture_output=True).returncode:
    pip = f'uv pip install --python {V1_PY}'
    sh(f'{pip} setuptools wheel ninja')
    sh(f'{pip} --no-build-isolation git+https://github.com/nerfstudio-project/gsplat.git',
       env={**V1_ENV, 'TORCH_CUDA_ARCH_LIST': f'{COMPUTE_CAP:.1f}', 'MAX_JOBS': str(os.cpu_count() or 2),
            'CUDA_HOME': os.environ.get('CUDA_HOME', '/usr/local/cuda')})
check_cuda(V1_PY)
# The depth-sensor model uses xFormers attention; make sure its CUDA kernels load and run.
XFORMERS_CHECK = ("import torch, xformers, xformers.ops as xo; "
                  "q = torch.randn(1, 256, 8, 64, device='cuda', dtype=torch.half); xo.memory_efficient_attention(q, q, q); "
                  "print('xFormers', xformers.__version__, 'memory-efficient attention OK on the GPU')")
sh(f'{V1_PY} -c {shlex.quote(XFORMERS_CHECK)}')
sh(f'{V1_PY} -c "from src.model.decoder.decoder_gsplat import is_gsplat_available as ok; '
   'print(\'gsplat (MP4 video):\', ok())"', cwd=V1_DIR)
sh('"$SPLAT_TRANSFORM" --version', env=V1_ENV)

In [ ]:
#@title B.2 · Download the v1 checkpoints
DOWNLOAD_RGB = True  #@param {type:"boolean"}
DOWNLOAD_DEPTH_SENSOR = True  #@param {type:"boolean"}
#@markdown RGB checkpoint: 3.1 GB (cells B.3 and B.5). Depth-sensor checkpoint: 2.7 GB (cell B.4 and B.5 with a depth map).
require(V1_PY, 'B.1')
for wanted, name in ((DOWNLOAD_RGB, 'infinisplat_rgb.ckpt'), (DOWNLOAD_DEPTH_SENSOR, 'infinisplat_lidar.ckpt')):
    if wanted and not (V1_DIR / 'checkpoints' / name).is_file():
        sh(f'hf download PLUS-WAVE/InfiniSplat checkpoints/{name} --local-dir .', cwd=V1_DIR, env=V1_ENV)
list_files(V1_DIR / 'checkpoints')

In [ ]:
#@title B.3 · Single RGB image (bundled examples)
EXAMPLE_IMAGE = 'pexels-masi.jpg'  #@param ["animate_room.jpg", "bedroom.jpg", "beggar_home.jpg", "cave_ai.jpg", "dragon_ball.jpg", "eth3d_courtyard.png", "flower_room.jpg", "ghibli_realroom.jpg", "ghibli_room.jpg", "gym.png", "living_room.jpg", "loft_room.jpg", "maksim-shutov-unsplash.jpg", "meerkat.jpg", "my_bedroom.JPG", "old_livingroom.png", "painting_room.jpg", "pexels-masi.jpg", "scannetpp_fe94fc30cf.JPG", "sofa_ai.jpg", "summer_room.jpg"]
FOCAL_MM = 0  #@param {type:"number"}
#@markdown `FOCAL_MM` is the 35 mm-equivalent focal length; `0` uses the image's EXIF data, or 30 mm when there is none.

image = V1_DIR / 'examples' / 'data' / 'rgb_demo' / EXAMPLE_IMAGE
LAST_V1 = run_v1(image, mode='rgb', focal_mm=FOCAL_MM)
show_v1(LAST_V1, [image])

In [ ]:
#@title B.4 · RGB image + depth-sensor map (bundled examples)
EXAMPLE_PAIR = 'eth3d_kicker'  #@param ["eth3d_kicker", "eth3d_pipes", "waymo_147", "waymo_9"]
#@markdown ETH3D examples carry dense depth; Waymo examples carry sparse LiDAR points (`mask` + `value`).

image = V1_DIR / 'examples' / 'data' / 'lidar_demo' / f'{EXAMPLE_PAIR}.png'
LAST_V1 = run_v1(image, mode='lidar')
show_v1(LAST_V1, [image], depth=image.with_suffix('.npz'))

In [ ]:
#@title B.5 · Your own image(s)
IMAGE_PATH = ''  #@param {type:"string"}
FOCAL_MM = 0  #@param {type:"number"}
DEPTH_PATH = ''  #@param {type:"string"}
#@markdown * Leave `IMAGE_PATH` empty to upload one or more images, or give an image file or a folder. JPG, PNG, BMP and WebP are supported; each image is reconstructed on its own.
#@markdown * `DEPTH_PATH` (optional, single image only) switches to the depth-sensor model. The map must be aligned with the image, use larger values for farther points, have valid values between 1 and 100 (`.npy`, `.npz` with `data` or sparse `mask`/`value`, `.h5`, `.exr`). See `InfiniSplat_v1/docs/inference.md`.

source = Path(IMAGE_PATH) if IMAGE_PATH else upload_files('v1')
images = [source] if source.is_file() else list_images(source, IMAGE_EXTS | {'.bmp'})
if not images:
    raise ValueError(f'No images found in {source}.')
if DEPTH_PATH and len(images) != 1:
    raise ValueError('DEPTH_PATH applies to a single image; set IMAGE_PATH to one file.')
LAST_V1 = run_v1(source, mode='lidar' if DEPTH_PATH else 'rgb', focal_mm=FOCAL_MM, prompt_depth=DEPTH_PATH or None)
show_v1(LAST_V1, images, depth=DEPTH_PATH or None)

In [ ]:
#@title B.6 · Download the latest v1 result (zip)
download(LAST_V1)

## Keep your results

Everything is written under `/content/outputs` (`v2/` and `v1/`, one new folder per run), which disappears when the Colab runtime is recycled. Download results with cells A.6 and B.6, or copy them all to Google Drive:

In [ ]:
#@title Copy all results to Google Drive
DRIVE_FOLDER = 'InfiniSplat_outputs'  #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
target = Path('/content/drive/MyDrive') / DRIVE_FOLDER
shutil.copytree(OUTPUTS, target, dirs_exist_ok=True, ignore=shutil.ignore_patterns('uploads'))
print('Copied to', target)

## Troubleshooting

| Symptom | What to do |
| --- | --- |
| `CUDA OK` is missing, or a run prints *barely used the GPU* | Choose *Runtime → Change runtime type → A100 GPU*, then *Runtime → Disconnect and delete runtime* and run again from 0.1. If the install cell says PyTorch cannot use the GPU on an older driver, rerun it with `TORCH_CUDA = cu126`. |
| `CUDA out of memory` | With `NUM_VIEWS = 0` v2 retries with fewer views by itself. Otherwise lower `NUM_VIEWS`, then `LONG_SIDE`. On a T4, start with 4 views at 280 px. |
| Exit code `-9` / `137`, or the runtime restarts | Out of system RAM while loading weights. Use *Runtime → Change runtime type → High-RAM*, or restart the runtime and rerun only the cells you need. |
| `NameError` after a runtime restart | Rerun cells 0.1–0.3. Installed environments and downloaded weights are kept on disk until the runtime is recycled. |
| An install cell failed halfway | Run it again; it starts the environment from scratch until it succeeds. |
| The viewer stays blank | Large scenes take a while to load; use the *Open the viewer in a new browser tab* link, or download `scene.html` and open it locally. The point previews never depend on the viewer. |
| Poor v2 reconstruction | Check overlap between photos, motion blur and exposure. Calibrated cameras (`provided`) are usually best; then `da3-pose`. |
| Poor v1 reconstruction | Set `FOCAL_MM` if EXIF data is missing; the 30 mm fallback can distort geometry. |

More details: [v2 inference guide](https://github.com/Cortex78/InfiniSplat/blob/main/InfiniSplat_v2/docs/inference.md) · [v1 inference guide](https://github.com/Cortex78/InfiniSplat/blob/main/InfiniSplat_v1/docs/inference.md)